In [1]:
import duckdb
import yaml
import os
import requests
import pandas as pd
import uuid
import json
from datetime import datetime, timezone

from shared_code.database.directory_creation import *
from shared_code.database.database_and_schema_creation import *
from pipelines.mojang_version_manifest.schemas.mojang_schemas import *

In [2]:
def get_official_minecraft_versions(url: str, headers: dict) -> dict:
    
    print(f"\t (INFO) Attempting to connect to: {url}")
    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=30,
        )
        response.raise_for_status()

        payload = response.json()
    
    except Exception as e:
        print(f"\t(ERROR) Error occurred processing request: {e}")
        return None

    return payload

In [3]:
def start_ingestion_log(
    db_con: duckdb.DuckDBPyConnection,
    ingestion_log: str,
    source_url: str,
    run_id: str,
    ingestion_type: str,
    project_type: str,
) -> None:

    insert_sql = f"""
        INSERT INTO {ingestion_log}
        (
            run_id,
            ingestion_type,
            api_url,
            project_type,
            status,
            start_time
        )
        VALUES (?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
    """

    db_con.execute(
        insert_sql,
        [
            run_id,
            ingestion_type,
            source_url,
            project_type,
            "running",
        ],
    )

In [4]:
def finish_ingestion_log(
    db_con: duckdb.DuckDBPyConnection,
    ingestion_log: str,
    run_id: str,
    ingestion_type: str,
    project_type: str,
    status: str,
    records_processed: int = 0,
    records_written: int = 0,
    records_failed: int = 0,
    records_skipped: int = 0,
    nested_records_fetched: int | None = None,
    failed_record_ids: list[str] | None = None,
    skipped_record_ids: list[str] | None = None,
    error_message: str | None = None,
) -> None:

    update_sql = f"""
        UPDATE {ingestion_log}
        SET
            status = ?,
            records_processed = ?,
            records_written = ?,
            records_failed = ?,
            records_skipped = ?,
            nested_records_fetched = ?,
            failed_record_ids = ?,
            skipped_record_ids = ?,
            error_message = ?,
            end_time = CURRENT_TIMESTAMP,
            duration_seconds = EXTRACT(
                EPOCH FROM (
                    CURRENT_TIMESTAMP - start_time
                )
            )
        WHERE run_id = ?
          AND ingestion_type = ?
          AND project_type = ?
    """

    db_con.execute(
        update_sql,
        [
            status,
            records_processed,
            records_written,
            records_failed,
            records_skipped,
            nested_records_fetched,
            failed_record_ids,
            skipped_record_ids,
            error_message,
            run_id,
            ingestion_type,
            project_type,
        ],
    )

In [5]:
stream_config = "stream-config.yml"
with open(stream_config, 'r') as file:
    config_data = yaml.safe_load(file)
    
config = config_data.get("main", {})
streams = config_data.get("streams", [])
headers = config.get("headers", {})
#layer and env to be populated via orchestrator
layer = 'bronze'
env = 'dev'
project_root = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "..",
        "..",
    )
)
ingestion_log = config.get("ingestion-log", '')
file_name = build_db_filename(streams[0]['source-url'])
table_name = streams[0]['table-name']
bronze_path = os.path.join(project_root, config['parent-folder'], layer, env, file_name)

build_layer_directory(os.path.dirname(bronze_path))
bronze_schemas = [build_bronze_mojang_version_manifest_schema(table_name), build_ingestion_log_schema(ingestion_log)]
init_db(bronze_path, bronze_schemas)

In [ ]:
run_id = str(uuid.uuid4())
ingestion_type = "mojang_version_manifest"


with duckdb.connect(bronze_path) as bronze_con:

    for idx, stream in enumerate(streams):

        table_name = stream["table-name"]
        source_url = stream["source-url"]
        stream_name = stream["name"]

        print(
            f"{idx + 1}/{len(streams)} "
            f"Ingesting stream: {stream_name}"
        )

        start_ingestion_log(
            db_con=bronze_con,
            ingestion_log=ingestion_log,
            source_url=source_url,
            run_id=run_id,
            ingestion_type=ingestion_type,
            project_type=stream_name,
        )

        try:
            payload = get_official_minecraft_versions(
                url=source_url,
                headers=headers,
            )

            if not payload:
                raise ValueError(
                    f"Empty payload returned from {source_url}"
                )

            print("\t (INFO): Data retrieved")

            pull_timestamp_utc = datetime.now(timezone.utc)

            insert_sql = f"""
                INSERT INTO {table_name}
                (
                    run_id,
                    stream,
                    payload,
                    c_pull_timestamp_utc
                )
                VALUES (?, ?, ?, ?)
            """

            bronze_con.execute(
                insert_sql,
                (
                    run_id,
                    stream_name,
                    json.dumps(payload),
                    pull_timestamp_utc,
                ),
            )

            nested_records_fetched = len(
                payload.get("versions", [])
            )

            finish_ingestion_log(
                db_con=bronze_con,
                ingestion_log=ingestion_log,
                run_id=run_id,
                ingestion_type=ingestion_type,
                project_type=stream_name,
                status="success",
                records_processed=1,
                records_written=1,
                records_failed=0,
                records_skipped=0,
                nested_records_fetched=nested_records_fetched,
            )

            print(
                "\t (SUCCESS): Data successfully retrieved "
                "and written to:"
                f"\n\t\t{bronze_path}"
            )

        except Exception as exc:

            finish_ingestion_log(
                db_con=bronze_con,
                ingestion_log=ingestion_log,
                run_id=run_id,
                ingestion_type=ingestion_type,
                project_type=stream_name,
                status="failed",
                records_processed=1,
                records_written=0,
                records_failed=1,
                records_skipped=0,
                nested_records_fetched=0,
                error_message=str(exc),
            )

            print(
                f"\t (ERROR): Failed to ingest "
                f"{stream_name}: {exc}"
            )

            raise

1/1 Ingesting stream: mojang_version_manifest
	 (INFO) Attempting to connect to: https://piston-meta.mojang.com/mc/game/version_manifest_v2.json
	 (INFO) Data retrieved
	 (SUCCESS) Data successfully retrieved and written to:
		/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||piston-meta.mojang.com.duckdb
